In [ ]:
mkdir data

mkdir: cannot create directory ‘data’: File exists


In [ ]:
%pip uninstall -y transformers tokenizers trl accelerate peft bitsandbytes tf-keras tensorflow
%pip install -U pip

# Stable HF stack for TRL SFTTrainer
%pip install "transformers==4.44.2" "trl==0.8.6" "accelerate==0.33.0" "peft==0.12.0" "datasets>=2.18.0" "sentencepiece" "bitsandbytes==0.43.1"

Found existing installation: transformers 4.44.2
Uninstalling transformers-4.44.2:
  Successfully uninstalled transformers-4.44.2
Found existing installation: tokenizers 0.19.1
Uninstalling tokenizers-0.19.1:
  Successfully uninstalled tokenizers-0.19.1
Found existing installation: trl 0.8.6
Uninstalling trl-0.8.6:
  Successfully uninstalled trl-0.8.6
Found existing installation: accelerate 0.33.0
Uninstalling accelerate-0.33.0:
  Successfully uninstalled accelerate-0.33.0
Found existing installation: peft 0.12.0
Uninstalling peft-0.12.0:
  Successfully uninstalled peft-0.12.0
Found existing installation: bitsandbytes 0.43.1
Uninstalling bitsandbytes-0.43.1:
  Successfully uninstalled bitsandbytes-0.43.1
  Using cached transformers-4.44.2-py3-none-any.whl.metadata (43 kB)
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
  Using cached accelerate-0.33.0-py3-none-any.whl.metadata (18 kB)
  Using cached peft-0.12.0-py3-none-any.whl.metadata (13 kB)
  Using cached bitsandbytes-0.

In [ ]:
from datasets import load_dataset
from pathlib import Path

ds = load_dataset("text", data_files={"train": "data/train.txt"})["train"]
ds = ds.train_test_split(test_size=0.02, seed=0, shuffle=True)

train_ds = ds["train"]
val_ds   = ds["test"]

Path("data/train_split.txt").write_text("\n".join(train_ds["text"]) + "\n", encoding="utf-8")
Path("data/val_split.txt").write_text("\n".join(val_ds["text"]) + "\n", encoding="utf-8")


Generating train split: 0 examples [00:00, ? examples/s]

116481

In [ ]:
import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input="data/train_split.txt",
    model_prefix="data/tokenizer_hanzi",
    vocab_size=6500,              # pick your size
    model_type="unigram",         # good default
    character_coverage=0.9995,    # important for CJK
    bos_id=2, eos_id=3, unk_id=1, pad_id=0,
)


In [ ]:
from transformers import set_seed
from transformers import AutoConfig, AutoModelForCausalLM, DebertaV2Tokenizer
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
import torch
import os

def preprocess_logits_for_metrics(logits, labels):
    """Extract predicted token IDs from model logits for evaluation"""
    pred_ids = torch.argmax(logits, dim=-1)  # Get the token with highest probability
    return pred_ids


def compute_metrics(eval_pred):
    """Calculate accuracy by comparing predictions with true labels"""
    logits, labels = eval_pred
    predictions = logits.flatten()  # Flatten to 1D array
    labels = labels.flatten()

    # Only consider non-padding tokens (labels != -100 are actual tokens)
    mask = labels != -100
    labels = labels[mask]
    predictions = predictions[mask]

    # Calculate accuracy
    correct = labels == predictions
    accuracy = correct.sum() / float(len(correct))
    return {"acc": accuracy}

In [ ]:
from transformers import DebertaV2Tokenizer

tokenizer = DebertaV2Tokenizer(vocab_file="data/tokenizer_hanzi.model")

# Ensure special tokens exist / are mapped
specials = {}
if tokenizer.pad_token is None: specials["pad_token"] = "[PAD]"
if tokenizer.eos_token is None: specials["eos_token"] = "</s>"
if tokenizer.bos_token is None: specials["bos_token"] = "<s>"
if specials:
    tokenizer.add_special_tokens(specials)


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
from datasets import load_dataset

train_ds = load_dataset("text", data_files={"train": "data/train_split.txt"})["train"]
val_ds   = load_dataset("text", data_files={"validation": "data/val_split.txt"})["validation"]


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [ ]:
from transformers import DebertaV2Tokenizer

tok = DebertaV2Tokenizer(vocab_file="data/tokenizer_hanzi.model")

print("len(tok) =", len(tok))
print("specials:", tok.pad_token, tok.bos_token, tok.eos_token)
print("ids:", tok.pad_token_id, tok.bos_token_id, tok.eos_token_id)

s = "已经很晚了，我们回家吧。"
enc = tok(s)
print("tokens:", tok.convert_ids_to_tokens(enc["input_ids"])[:40])
print("decoded:", tok.decode(enc["input_ids"]))


len(tok) = 6505
specials: [PAD] [CLS] [SEP]
ids: 6503 6500 6501
tokens: ['[CLS]', '▁', '<unk>', '[SEP]']
decoded: [CLS]  ⁇ [SEP]


In [ ]:
from transformers import GPT2Config, AutoModelForCausalLM

config = GPT2Config(
    vocab_size=len(tok),
    n_embd=384,
    n_inner=1280,
    n_layer=8,
    n_head=12,
    n_positions=1024,
    bos_token_id=tok.bos_token_id,
    eos_token_id=tok.eos_token_id,
    pad_token_id=tok.pad_token_id,
)
model = AutoModelForCausalLM.from_config(config)
model.resize_token_embeddings(len(tok))


Embedding(6505, 384)

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")


True
1
Tesla T4


In [ ]:
# working model
# from datasets import load_dataset
# from transformers import (
#     Trainer, TrainingArguments,
#     DataCollatorForLanguageModeling,
#     set_seed
# )
# import math

# set_seed(0)

# train_ds = load_dataset("text", data_files={"train": "data/train_split.txt"})["train"]
# val_ds   = load_dataset("text", data_files={"validation": "data/val_split.txt"})["validation"]

# def tokenize_fn(examples):
#     return tok(examples["text"], truncation=True, max_length=128)

# train_tok = train_ds.map(tokenize_fn, batched=True, num_proc=12, remove_columns=["text"])
# val_tok   = val_ds.map(tokenize_fn, batched=True, num_proc=12, remove_columns=["text"])

# collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)

# training_args = TrainingArguments(
#     output_dir="outputs/hanzi_sft",
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     num_train_epochs=1,
#     evaluation_strategy="steps",
#     eval_steps=500,
#     save_steps=500,
#     logging_steps=50,
#     report_to="none",
# )

# def compute_metrics(eval_pred):
#     # Trainer provides loss separately via eval_loss in logs,
#     # so metrics are optional. We'll compute ppl from eval_loss later.
#     return {}

# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tok,
#     train_dataset=train_ds,
#     eval_dataset=val_ds,
#     args=training_args,
#     max_seq_length=128,
#     dataset_text_field="text",
# )

# trainer.train()


In [ ]:
# corrected
from datasets import load_dataset
from transformers import (
    Trainer, TrainingArguments,
    DataCollatorForLanguageModeling,
    set_seed
)
import math

set_seed(0)

train_ds = load_dataset("text", data_files={"train": "data/train_split.txt"})["train"]
val_ds   = load_dataset("text", data_files={"validation": "data/val_split.txt"})["validation"]

def tokenize_fn(examples):
    return tok(examples["text"], truncation=True, max_length=128)

train_tok = train_ds.map(tokenize_fn, batched=True, num_proc=12, remove_columns=["text"])
val_tok   = val_ds.map(tokenize_fn, batched=True, num_proc=12, remove_columns=["text"])

collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)

args = TrainingArguments(
    output_dir="outputs/hanzi_baseline",
    per_device_train_batch_size=8,     # CPU: start small
    use_cpu=False,
    # fp16=True,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,     # effective batch 64
    num_train_epochs=10,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=3,
    report_to="none",
    fp16=False,
    bf16=False,
    # dataloader_num_workers=0,          # safer on some CPU envs
)

def compute_metrics(eval_pred):
    # Trainer provides loss separately via eval_loss in logs,
    # so metrics are optional. We'll compute ppl from eval_loss later.
    return {}

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
    tokenizer=tok,   # keeps logs consistent
    # processing_class=tok,
)

trainer.train()


Map (num_proc=12):   0%|          | 0/14659 [00:00<?, ? examples/s]

Map (num_proc=12):   0%|          | 0/300 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss
500,5.932300,5.917016
1000,5.337700,5.452871
1500,5.029700,5.251553
2000,4.855600,5.182566


TrainOutput(global_step=2290, training_loss=5.50381970134885, metrics={'train_runtime': 706.3904, 'train_samples_per_second': 207.52, 'train_steps_per_second': 3.242, 'total_flos': 996445127122944.0, 'train_loss': 5.50381970134885, 'epoch': 9.99454446262957})

In [ ]:
import torch
from transformers import AutoModelForCausalLM, DebertaV2Tokenizer

ckpt = "outputs/hanzi_baseline/checkpoint-2290"  # or full path
tokenizer = DebertaV2Tokenizer(vocab_file=f"{ckpt}/spm.model")
model = AutoModelForCausalLM.from_pretrained(ckpt, torch_dtype=torch.float16, device_map="auto")

prompt = "jīntiān tiānqì"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

out = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.9,
    top_p=0.95,
    repetition_penalty=1.1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

print(tokenizer.decode(out[0], skip_special_tokens=True))

jīntiān tiānqì dōu míngyī dàjiǎngtáng dàocǐjiéshù rúguǒ nín xiǎng liǎojiě gèng duō jiànkāng kēpǔzhīshí měizhōu yīdào zhōuwǔ xiàwǔ nán dōu ài jiànkāng wēixìn gōngzhòng hào tóutiáo hào hé xīnlàng wēibóhào xǐmǎlāyǎ děng píngtái péinín dùguò měimǎn jiànkāng de yītiān wǒmen xiàqī zàijiàn


In [ ]:
# !python train_hanzi_baseline.py --corpus corpus.txt --epochs 10